# Detección local de objetos en tiempo real con YOLOv8

**Estudiante:** Nombre Apellido  
**Tema:** Redes neuronales convolucionales y detección de objetos  
**Modelo:** YOLOv8 Nano preentrenado con el conjunto de datos COCO

Este notebook se ejecuta **localmente** en Jupyter Notebook, JupyterLab o VS Code. Utiliza OpenCV para leer la cámara de la computadora y YOLOv8 Nano para detectar varios objetos en tiempo real.

Cada detección muestra:

- Recuadro delimitador.
- Nombre del objeto.
- Nivel de confianza.

> La primera ejecución necesita conexión a Internet únicamente para instalar las bibliotecas y descargar automáticamente el modelo `yolov8n.pt`.

## Instrucciones de uso

1. Abra este archivo localmente con Jupyter Notebook, JupyterLab o VS Code.
2. Ejecute las celdas en orden.
3. Al ejecutar una prueba aparecerá una ventana llamada **Detección local - YOLOv8**.
4. Haga clic sobre esa ventana para darle el foco.
5. Presione **C** para capturar la evidencia actual.
6. Presione **Q** o **Esc** para cerrar sin capturar.
7. Ejecute las tres pruebas y conserve sus imágenes visibles en el notebook.

Si Windows solicita permiso para usar la cámara, seleccione **Permitir**.

## 1. Instalación de bibliotecas

Esta celda instala YOLOv8 y OpenCV dentro del mismo entorno de Python utilizado por el kernel del notebook.

In [ ]:
# Instala versiones CPU compatibles de PyTorch y Torchvision.
# Solo reemplaza estas bibliotecas si las versiones locales no coinciden.
%pip install -q --upgrade --no-deps "torch==2.11.0+cpu" "torchvision==0.26.0+cpu" --index-url https://download.pytorch.org/whl/cpu

# Elimina OpenCV Headless porque no permite abrir ventanas con cv2.imshow.
%pip uninstall -y opencv-python-headless

# Reinstala OpenCV con interfaz gráfica y después instala el detector.
%pip install -q --upgrade --force-reinstall --no-deps "opencv-python>=4.8,<5"
%pip install -q "ultralytics>=8.3,<9" pandas pillow

print("Bibliotecas instaladas correctamente.")
print("Reinicie el kernel después de ejecutar esta celda y continúe en la sección 2.")

## 2. Importación de librerías y carga del modelo

YOLOv8 Nano es una versión ligera apropiada para trabajar en tiempo real. Reconoce las 80 categorías del conjunto COCO, entre ellas persona, teléfono celular, botella, silla, mochila y computadora portátil.

In [ ]:
# Librerías utilizadas para la cámara, inferencia y visualización.
import os
import platform
from pathlib import Path

# Guarda la configuración de Ultralytics junto al notebook y evita problemas
# de permisos en carpetas globales del usuario.
carpeta_configuracion = Path.cwd() / ".yolo_config"
carpeta_configuracion.mkdir(exist_ok=True)
os.environ.setdefault("YOLO_CONFIG_DIR", str(carpeta_configuracion))

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display, Markdown
from ultralytics import YOLO

# Descarga y carga automática de los pesos preentrenados la primera vez.
modelo = YOLO("yolov8n.pt")

# Utiliza GPU si está disponible; de lo contrario, trabaja con el procesador.
dispositivo = 0 if torch.cuda.is_available() else "cpu"

print("Modelo YOLOv8 Nano cargado correctamente.")
print(f"Sistema operativo: {platform.system()}")
print(f"Dispositivo de inferencia: {'GPU' if dispositivo == 0 else 'CPU'}")
print(f"Cantidad de clases reconocibles: {len(modelo.names)}")

## 3. Funciones para abrir la cámara y detectar objetos

`cv2.VideoCapture(0)` accede directamente a la cámara local. Cada fotograma pasa por YOLOv8 y se muestra inmediatamente en una ventana de OpenCV.

Cuando se presiona **C**, la función conserva el fotograma procesado, muestra la evidencia dentro del notebook y genera una tabla con todos los objetos encontrados.

In [ ]:
def abrir_camara(indice=0):
    """Abre una cámara local y usa un método alternativo si es necesario."""
    # DirectShow suele funcionar mejor con cámaras locales en Windows.
    if os.name == "nt":
        camara = cv2.VideoCapture(indice, cv2.CAP_DSHOW)
        if not camara.isOpened():
            camara.release()
            camara = cv2.VideoCapture(indice)
    else:
        camara = cv2.VideoCapture(indice)

    # Solicita resolución HD; la cámara puede seleccionar una cercana.
    camara.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    camara.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    return camara


def construir_tabla(resultado):
    """Convierte las cajas detectadas por YOLO en una tabla legible."""
    filas = []
    cajas = resultado.boxes

    if cajas is not None:
        for numero, (clase, confianza, coordenadas) in enumerate(
            zip(cajas.cls.tolist(), cajas.conf.tolist(), cajas.xyxy.tolist()),
            start=1,
        ):
            x1, y1, x2, y2 = [round(valor) for valor in coordenadas]
            filas.append({
                "N.º": numero,
                "Objeto": modelo.names[int(clase)],
                "Confianza": f"{confianza * 100:.1f} %",
                "Recuadro (x1, y1, x2, y2)": f"({x1}, {y1}, {x2}, {y2})",
            })

    return pd.DataFrame(filas)


def detectar_en_tiempo_real(nombre_prueba, confianza_minima=0.30, indice_camara=0):
    """
    Detecta objetos continuamente con la cámara local.
    C captura la evidencia actual; Q o Esc cierran sin capturar.
    """
    camara = abrir_camara(indice_camara)
    if not camara.isOpened():
        raise RuntimeError(
            "No fue posible abrir la cámara. Cierre otras aplicaciones que la usen, "
            "revise los permisos del sistema o pruebe indice_camara=1."
        )

    ventana = "Detección local - YOLOv8"
    evidencia = None

    print(f"Iniciando: {nombre_prueba}")
    print("En la ventana de video presione C para capturar, o Q/Esc para salir.")

    try:
        while True:
            lectura_correcta, fotograma = camara.read()
            if not lectura_correcta:
                raise RuntimeError("La cámara se abrió, pero no pudo entregar imágenes.")

            # El modelo analiza el fotograma y puede producir varias detecciones.
            resultado = modelo.predict(
                source=fotograma,
                conf=confianza_minima,
                imgsz=640,
                max_det=100,
                device=dispositivo,
                verbose=False,
            )[0]

            # plot() dibuja las cajas, las clases y los niveles de confianza.
            fotograma_anotado = resultado.plot(line_width=2, font_size=13)

            # Agrega instrucciones visibles en la parte superior de la ventana.
            cv2.rectangle(fotograma_anotado, (0, 0), (fotograma_anotado.shape[1], 42), (25, 25, 25), -1)
            cv2.putText(
                fotograma_anotado,
                "C = capturar evidencia    Q o Esc = salir",
                (12, 29),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.72,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

            cv2.imshow(ventana, fotograma_anotado)
            tecla = cv2.waitKey(1) & 0xFF

            if tecla in (ord("c"), ord("C")):
                # Se guardan copias para que no cambien al continuar el ciclo.
                evidencia = {
                    "nombre": nombre_prueba,
                    "imagen_bgr": fotograma_anotado.copy(),
                    "resultado": resultado,
                }
                break

            if tecla in (ord("q"), ord("Q"), 27):
                break
    finally:
        # Libera siempre la cámara, incluso si ocurre un error.
        camara.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)

    if evidencia is None:
        print("Prueba cerrada sin capturar evidencia.")
        return None

    # Convierte de BGR a RGB para mostrar correctamente la evidencia en Jupyter.
    imagen_rgb = cv2.cvtColor(evidencia["imagen_bgr"], cv2.COLOR_BGR2RGB)
    tabla = construir_tabla(evidencia["resultado"])

    display(Markdown(f"### Evidencia: {nombre_prueba}"))
    display(Image.fromarray(imagen_rgb))

    if tabla.empty:
        print("No se detectaron objetos con la confianza solicitada.")
        print("Repita la prueba con mejor iluminación o reduzca confianza_minima.")
    else:
        display(tabla.style.hide(axis="index"))
        print(f"Total de objetos detectados: {len(tabla)}")

    evidencia["tabla"] = tabla
    return evidencia


print("Funciones de detección local preparadas.")

## 4. Evidencias de las tres pruebas mínimas

Ejecute cada celda por separado. Prepare la escena, seleccione la ventana de la cámara y presione **C**. La imagen anotada y la tabla quedarán almacenadas en la salida del notebook.

### Prueba 1: persona frente a la cámara

Coloque a una persona frente a la cámara. El modelo COCO detecta la clase `person`; no detecta el rostro como una clase independiente.

In [ ]:
# Prueba 1: presione C cuando aparezca una persona detectada.
evidencia_prueba_1 = detectar_en_tiempo_real(
    "Prueba 1 — Persona frente a la cámara",
    confianza_minima=0.30,
)

### Prueba 2: objeto individual

Muestre un solo objeto reconocido por COCO, como una botella, teléfono celular, taza, mochila, libro o computadora portátil.

In [ ]:
# Prueba 2: presione C cuando aparezca el objeto individual detectado.
evidencia_prueba_2 = detectar_en_tiempo_real(
    "Prueba 2 — Objeto individual",
    confianza_minima=0.30,
)

### Prueba 3: dos o más objetos simultáneamente

Coloque al menos dos objetos en la escena; por ejemplo, una persona sosteniendo una botella y un teléfono.

In [ ]:
# Prueba 3: presione C cuando existan dos o más objetos detectados.
evidencia_prueba_3 = detectar_en_tiempo_real(
    "Prueba 3 — Dos o más objetos simultáneos",
    confianza_minima=0.25,
)

## 5. Verificación final

Ejecute esta celda después de capturar las tres evidencias.

In [ ]:
# Comprueba que cada prueba tenga una captura y suficientes detecciones.
nombres_variables = [
    "evidencia_prueba_1",
    "evidencia_prueba_2",
    "evidencia_prueba_3",
]

evidencias = [globals().get(nombre) for nombre in nombres_variables]

if all(evidencia is not None for evidencia in evidencias):
    cantidades = [len(evidencia["tabla"]) for evidencia in evidencias]
    print("✅ Las tres evidencias fueron capturadas.")
    print(f"Objetos detectados por prueba: {cantidades}")

    if cantidades[0] >= 1 and cantidades[1] >= 1 and cantidades[2] >= 2:
        print("✅ Se cumplen las detecciones mínimas solicitadas.")
    else:
        print("⚠️ Repita las pruebas que no tengan suficientes objetos detectados.")
else:
    faltantes = [
        nombre for nombre, evidencia in zip(nombres_variables, evidencias)
        if evidencia is None
    ]
    print("⚠️ Faltan evidencias por capturar:", ", ".join(faltantes))

## Solución de problemas

- **La cámara no abre:** cierre Zoom, Teams, Meet u otra aplicación que esté utilizando la cámara.
- **La cámara correcta no es la número 0:** agregue `indice_camara=1` en las llamadas de prueba.
- **No reconoce un objeto:** mejore la iluminación, acerque el objeto o reduzca `confianza_minima` a `0.20`.
- **La detección es lenta:** reduzca `imgsz=640` a `imgsz=480` dentro de la función.
- **La ventana parece bloqueada:** haga clic en ella antes de presionar C o Q.
- **El kernel no reconoce una biblioteca recién instalada:** reinicie el kernel y vuelva a ejecutar desde la sección 2.

## Conclusión

Se implementó localmente un sistema de visión artificial que obtiene video en tiempo real desde la cámara de la computadora. YOLOv8 Nano analiza cada fotograma, identifica varios objetos y presenta su clase, confianza y recuadro delimitador.

**Antes de entregar:** cambie `NombreApellido` en el nombre del archivo y en el encabezado, ejecute las tres pruebas y guarde el notebook conservando todas las salidas.